# AI Respondents Challenge — Final Pipeline (v4)

**Pipeline definition.** A gated two-expert system for predicting hidden survey answers.
A per-target gradient-boosting classifier trained on all allowed features is the **primary
expert**; a large language model, prompted with the respondent's most informative answers and
similar labeled respondents, is the **challenger**; a per-target **gate policy** — tuned on
held-out training respondents with the official skill metric — decides which expert answers
each (respondent, question) pair on the test set.

**Dataflow**

```
TRAIN (5,000 labeled respondents, 50 countries)
  │
  ├─ [A] GBM stage ─ one HistGradientBoosting model per target, all 278 features
  │       ├─ leave-countries-out CV  →  honest OOF predictions + confidences
  │       └─ full-train fit          →  test predictions + confidences
  │
  ├─ [B] Statistics stage ─ Cramér's V per (feature, target)
  │       ├─ top-12 features per target  →  what the LLM prompt shows
  │       └─ V values as weights         →  kNN index of look-alike respondents
  │
  ├─ [C] LLM selection stage (on train, answers known)
  │       ├─ phase 1: 3 models × 2 prompt styles on 250 respondents  →  winner
  │       └─ phase 2: winner on 1,000 respondents                    →  reliable per-target skill
  │
  └─ [D] Gate tuning ─ per target, best of {always-GBM, always-LLM, GBM-if-confident-else-LLM}
          scored on phase-2 respondents against OOF GBM  →  policy table

TEST (1,050 respondents, 13 hidden answers each)
  GBM predicts all 13,650 pairs ──> gate routes low-confidence pairs (~11%) to the frozen LLM
  ──> final assembly ──> calibration sanity check ──> predictions.csv + features.csv + method/
  ──> submission.zip
```

**Frozen final configuration:** Qwen3-235B-A22B-Instruct-2507, third-person prompt,
first-token logprob scoring; per-target policies as printed in section 8, with Q17/Q112
manually forced to GBM; Q112 labels re-binned after a calibration check exposed a
scale/label mismatch (section 10).

In [ ]:
%pip install -q datasets scikit-learn scipy openai

## 0. Configuration
Every tunable choice of the pipeline in one place, so this cell doubles as the frozen-method
record. The `FINAL_*` values pin the LLM configuration that won the offline bake-off.
`LOCAL_DATA_DIR` re-points the whole notebook at a different survey folder without any other
change; `EVAL_CACHE` makes every LLM evaluation call resumable and never paid for twice.

In [ ]:
RUN_LLM_EVAL = True    # phase 1+2 offline evaluation (tunes model/style choice and the gate)
RUN_LLM_TEST = True   # LLM on test where the gate requires it (final run)
N_COMPARE    = 250     # respondents for the model x style comparison (phase 1)
N_EVAL       = 1000    # respondents for final validation + gate tuning (phase 2, winner only)
K_SHOTS      = 3       # few-shot neighbour examples per prompt
TOP_FEATURES = 12      # features shown to the LLM per target
WORKERS      = 8
SEED         = 0

MODELS = ["Qwen/Qwen3-32B",
          "Qwen/Qwen3-235B-A22B-Instruct-2507",
          "meta-llama/Llama-3.3-70B-Instruct"]
STYLES = ["third", "first"]        # add "first_dist" to also test the distribution-anchored prompt

FINAL_MODEL = "Qwen/Qwen3-235B-A22B-Instruct-2507"
FINAL_STYLE = "third"
FINAL_ENSEMBLE = None

LOCAL_DATA_DIR = None                     # folder with train/test/targets/features.csv (held-out surveys)
TARGETS_EXTRA  = ["targets_hidden.csv"]   # extra target files, ignored if absent
EVAL_CACHE     = "eval_cache.csv"         # LLM eval results cache (survives re-runs)

## 1. Load data and build label maps
Loads the four challenge tables (train / test / targets / features), downloads and appends the
4 hidden targets, and derives everything downstream from them: the target list `qids`, the
code→label map per target (`opt2lab`, from the `option` column), the label order (`labels_for`),
and the allowed feature pool `fv` — with target questions force-excluded, since targets may
never be used as features. Expected printout: 13 targets, 278 features, 20 seen + 15 unseen
test countries.

In [ ]:
import json, os, warnings
import numpy as np
import pandas as pd
from datasets import load_dataset
warnings.filterwarnings("ignore")


from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="oxford-llms/ai-respondents-challenge",
                filename="targets_hidden.csv", repo_type="dataset",
                local_dir=".")

REPO = "oxford-llms/ai-respondents-challenge"
def load_table(name):
    if LOCAL_DATA_DIR:
        return pd.read_csv(os.path.join(LOCAL_DATA_DIR, name + ".csv"))
    return load_dataset(REPO, name, split="train").to_pandas()

train, test = load_table("train"), load_table("test")
targets, features = load_table("targets"), load_table("features")

for f in TARGETS_EXTRA:
    path = os.path.join(LOCAL_DATA_DIR or ".", f)
    if os.path.exists(path):
        extra = pd.read_csv(path)
        targets = pd.concat([targets, extra], ignore_index=True).drop_duplicates(
            subset=["question_id", "option"])
        print(f"added {extra.question_id.nunique()} extra targets from {f}")

qids = targets.question_id.unique().tolist()
assert not [q for q in qids if q not in train.columns], "target missing from train columns"

opt2lab    = {q: dict(zip(g.option, g.label)) for q, g in targets.groupby("question_id")}
labels_for = {q: [opt2lab[q][o] for o in sorted(opt2lab[q])] for q in qids}
question_for = dict(zip(targets.question_id, targets.question))

fv = [v for v in features.variable if v in train.columns and v not in qids]
dropped = [v for v in features.variable if v in qids]
if dropped: print("excluded targets from feature pool:", dropped)
qtext = dict(zip(features.variable, features.question))
vmaps = {v: json.loads(s) for v, s in zip(features.variable, features.values_json)}

train_lab = pd.DataFrame({q: train[q].map(opt2lab[q]) for q in qids})
seen = sorted(set(train.country) & set(test.country))
unseen = sorted(set(test.country) - set(train.country))
print(f"{len(qids)} targets, {len(fv)} features, {len(seen)} seen / {len(unseen)} unseen test countries")

targets_hidden.csv:   0%|          | 0.00/2.75k [00:00<?, ?B/s]

added 4 extra targets from targets_hidden.csv
13 targets, 278 features, 20 seen / 15 unseen test countries


## 2. Stage A — the GBM primary expert
One `HistGradientBoostingClassifier` per target, trained on all 278 features plus a country
code (unseen countries encode as unknown). Ordinal answer codes are used as-is and missing
answers are handled natively by the trees — no imputation. Each target is fitted twice:

- **OOF pass** — 5-fold cross-validation that holds out whole *countries*, mimicking the 15
  test countries never seen in training. The out-of-fold predictions and confidences are the
  honest yardstick the gate is tuned against in section 8.
- **Full fit** — trained on all 5,000 respondents; produces the test predictions and the
  per-prediction confidence (max class probability) that the gate thresholds at test time.

Reading the output: `majority` is the share of the most common answer (the zero-skill
baseline), and `OOF skill` is the official metric — how far above majority-guessing the model
lands with countries held out. Note Q112's implausibly low +0.03: this is the first symptom of
the label mismatch diagnosed and repaired in section 10.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

train["_c"] = train.country.astype("category").cat.codes
cat = train.country.astype("category").cat.categories
test["_c"] = pd.Categorical(test.country, categories=cat).codes
COLS = fv + ["_c"]

def new_model():
    return HistGradientBoostingClassifier(max_iter=200, early_stopping=True, random_state=SEED)
def skill(acc, maj): return (acc - maj) / (1 - maj)

gbm_oof_pred, gbm_oof_prob, gbm_full = {}, {}, {}
print(f"{'target':6s} {'majority':>9s} {'OOF acc':>8s} {'OOF skill':>10s}")
for q in qids:
    y = train_lab[q]; mask = y.notna()
    X = train.loc[mask, COLS].astype(float); yy = y[mask].values
    groups = train.loc[mask, "country"].values
    maj = (yy == pd.Series(yy).mode()[0]).mean()
    oof_pred = pd.Series(index=X.index, dtype=object)
    oof_prob = pd.Series(index=X.index, dtype=float)
    for tr, te in GroupKFold(n_splits=5).split(X, yy, groups):
        m = new_model().fit(X.iloc[tr], yy[tr])
        P = m.predict_proba(X.iloc[te])
        oof_pred.iloc[te] = m.classes_[P.argmax(1)]
        oof_prob.iloc[te] = P.max(1)
    gbm_oof_pred[q], gbm_oof_prob[q] = oof_pred, oof_prob
    acc = (oof_pred.values == yy).mean()
    print(f"{q:6s} {maj:9.1%} {acc:8.1%} {skill(acc, maj):+10.3f}")
    gbm_full[q] = new_model().fit(X, yy)

Xte = test[COLS].astype(float)
gbm_test = pd.DataFrame(index=test.index)
for q in qids:
    P = gbm_full[q].predict_proba(Xte)
    gbm_test[q + "_pred"] = gbm_full[q].classes_[P.argmax(1)]
    gbm_test[q + "_prob"] = P.max(1)
print("\nGBM test predictions ready.")

target  majority  OOF acc  OOF skill
Q201       40.5%    46.3%     +0.099
Q73        36.0%    73.7%     +0.589
Q227       27.6%    58.1%     +0.421
Q209       45.7%    70.5%     +0.457
Q33        31.6%    48.7%     +0.251
Q148       36.1%    71.8%     +0.558
Q17        67.2%    72.2%     +0.153
Q186       53.9%    69.8%     +0.344
Q242       38.3%    50.2%     +0.192
Q172       32.4%    52.9%     +0.303
Q22        54.9%    78.9%     +0.531
Q112       43.0%    44.7%     +0.030
Q34        35.8%    49.5%     +0.214

GBM test predictions ready.


## 3. Stage B — per-target feature statistics (Cramér's V)
An LLM prompt cannot fit 278 answers, so for each target we measure which features actually
predict it. Cramér's V is a 0-to-1 association measure between two categorical variables
(bias-corrected chi-square, comparable across different option counts); continuous features are
quintile-binned first. Per target we keep the top 12 features with at least 70% answer coverage
in the test set. The V values themselves are also retained (`FEATURE_W`) as distance weights for
the kNN retrieval in section 4. The printout shows the WVS block structure this recovers:
confidence items predict confidence in parliament, media-use items predict newspaper use, etc.

In [ ]:
from scipy.stats import chi2_contingency

def as_cat(s):
    s = s.dropna()
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 12:
        return pd.qcut(s, 5, duplicates="drop").astype(str)
    return s.astype(str)

def cramers_v(x, y):
    df = pd.concat([x, y], axis=1).dropna()
    if len(df) < 200: return np.nan
    a = as_cat(df.iloc[:, 0]); b = df.iloc[:, 1].loc[a.index]
    ct = pd.crosstab(a, b)
    if min(ct.shape) < 2: return np.nan
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum(); r, k = ct.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1); kc = k - (k - 1) ** 2 / (n - 1)
    d = min(kc - 1, rc - 1)
    return np.sqrt(phi2 / d) if d > 0 else np.nan

cov_test = test[fv].notna().mean()
PROMPT_FEATURES, FEATURE_W = {}, {}
for q in qids:
    y = train_lab[q]
    V = pd.Series({v: cramers_v(train[v], y) for v in fv}).dropna()
    V = V[cov_test.reindex(V.index) >= 0.70].sort_values(ascending=False)
    PROMPT_FEATURES[q] = V.head(TOP_FEATURES).index.tolist()
    FEATURE_W[q] = V.head(TOP_FEATURES).values          # kNN weights
    print(q, PROMPT_FEATURES[q][:5], "...")

Q201 ['Q203', 'Q202', 'Q205', 'Q200', 'Q204'] ...
Q73 ['Q72', 'Q71', 'Q74', 'Q76', 'Q70'] ...
Q227 ['Q230', 'Q231', 'Q226', 'Q225', 'Q224'] ...
Q209 ['Q218', 'Q211', 'Q210', 'Q212', 'Q217'] ...
Q33 ['Q31', 'Q29', 'Q30', 'Q35', 'Q20'] ...
Q148 ['Q147', 'Q146', 'Q143', 'Q142', 'Q57'] ...
Q17 ['Q8', 'Q164', 'Q45', 'Q27', 'Q6'] ...
Q186 ['Q193', 'Q183', 'Q187', 'Q185', 'Q188'] ...
Q242 ['Q243', 'Q245', 'Q246', 'Q239', 'Q241'] ...
Q172 ['Q165', 'Q168', 'Q167', 'Q15', 'Q166'] ...
Q22 ['Q20', 'Q182', 'Q25', 'Q36', 'Q289CS9'] ...
Q112 ['Q120', 'Q113', 'Q110', 'Q57', 'Q109'] ...
Q34 ['Q38', 'Q35', 'Q37', 'Q29', 'Q31'] ...


## 4. V-weighted kNN — retrieving look-alike respondents
For every (respondent, target) the prompt includes real training respondents with known answers
as few-shot examples. "Similar" is measured in the space of that target's 12 prompt features,
standardized and weighted by √V — so agreement on strongly predictive features counts more than
agreement on incidental ones. NaNs are median-imputed for distance purposes only. At evaluation
time a respondent is excluded from being their own neighbour; neighbours who did not answer the
target are skipped.

In [ ]:
from sklearn.neighbors import NearestNeighbors

knn_index, knn_scaler = {}, {}
def _matrix(df, q):
    cols = PROMPT_FEATURES[q]
    M = df[cols].astype(float)
    med, std = knn_scaler[q]
    Z = (M.fillna(med) - med) / std
    return (Z * np.sqrt(FEATURE_W[q])).values      # weight by sqrt(V): squared distance ~ V

for q in qids:
    M = train[PROMPT_FEATURES[q]].astype(float)
    med = M.median(); std = M.std().replace(0, 1)
    knn_scaler[q] = (med, std)
    knn_index[q] = NearestNeighbors(n_neighbors=K_SHOTS + 1).fit(_matrix(train, q))

def neighbours(resp_row_df, q, exclude_self_idx=None):
    d, idx = knn_index[q].kneighbors(_matrix(resp_row_df, q))
    out = [i for i in idx[0] if exclude_self_idx is None or i != exclude_self_idx]
    out = [i for i in out if pd.notna(train_lab[q].iloc[i])]
    return out[:K_SHOTS]

## 5. Nebius client and model availability
Creates the OpenAI-compatible client and verifies every model id in the grid actually exists on
the endpoint before any money is spent (a wrong id would otherwise fail 19,500 calls deep).

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

if "NEBIUS_API_KEY" not in os.environ:
    os.environ["NEBIUS_API_KEY"] = getpass("Nebius API key: ")
client = OpenAI(base_url="https://api.studio.nebius.com/v1/",
                api_key=os.environ["NEBIUS_API_KEY"])

available = {m.id for m in client.models.list().data}
for m in MODELS:
    print(f"{'OK ' if m in available else 'MISSING -> fix MODELS: '}{m}")

OK Qwen/Qwen3-32B
OK Qwen/Qwen3-235B-A22B-Instruct-2507
OK meta-llama/Llama-3.3-70B-Instruct


## 6. Prompt construction and the logprob scorer
Two prompt styles compete in section 7: `third` (a neutral analyst is asked what this respondent
most likely chose) and `first` (the model roleplays the respondent: "You are a survey respondent
from X … how do you answer?"). A `first_dist` variant that adds the country's training answer
distribution exists but was not part of the final grid. Every prompt contains: the respondent's
answers to the target's 12 features (in words, via the official code→text maps), 3 kNN
look-alikes with their true answers, the target question, and the numbered official labels.

Scoring: the model must reply with an option number; we read the **top-20 logprobs of the first
generated token** and collect the probability mass on each digit. This yields a full probability
distribution per call — the argmax is the prediction, the max is the confidence used by the
gate — and cannot suffer text-parsing failures. Fallbacks: parse the reply text, else uniform.
`llm_predict` also carries the operational armor added during the run: exponential-backoff
retries on 429 rate limits, and `enable_thinking: False` sent only to Qwen models that support
the switch (reasoning tokens would break first-token scoring).

In [ ]:
# per-(target, country) answer distributions from train, for the first_dist style
dist_country, dist_global = {}, {}
for q in qids:
    dist_global[q] = train_lab[q].value_counts(normalize=True)
    for c in seen:
        d = train_lab.loc[train.country == c, q].value_counts(normalize=True)
        if len(d): dist_country[(q, c)] = d

def dist_line(q, country):
    d = dist_country.get((q, country), dist_global[q])
    where = f"in {country}" if (q, country) in dist_country else "across all surveyed countries"
    parts = ", ".join(f"{l}: {d.get(l, 0):.0%}" for l in labels_for[q])
    return f"For context, the distribution of answers to this question {where} was: {parts}."

def answer_text(var, value):
    code = str(int(value)) if float(value).is_integer() else str(value)
    return vmaps[var].get(code, code)

def profile_lines(row, q, first_person=False):
    lead = "- " if not first_person else "- "
    return "\n".join(f"{lead}{qtext[v]} {answer_text(v, row[v])}"
                      for v in PROMPT_FEATURES[q] if pd.notna(row[v]))

def shots_block(nb_idx, q):
    s = ""
    for j, i in enumerate(nb_idx):
        r = train.iloc[i]
        s += (f"\nSimilar respondent {j+1} (from {r['country']}):\n"
              f"{profile_lines(r, q)}\n"
              f"Their answer to the question below: {train_lab[q].iloc[i]}\n")
    return s

def build_prompt(resp, q, nb_idx, style):
    opts = "\n".join(f"{i+1}. {l}" for i, l in enumerate(labels_for[q]))
    if style == "third":
        return (f"You predict how individual survey respondents answered a question, "
                f"based on their other answers.\n"
                f"{shots_block(nb_idx, q)}\n"
                f"Now the respondent to predict, from {resp['country']}:\n"
                f"{profile_lines(resp, q)}\n\n"
                f"Question: {question_for[q]}\nOptions:\n{opts}\n"
                f"Reply with only the number of the option this respondent most likely chose.")
    # first-person variants
    extra = f"\n{dist_line(q, resp['country'])}\n" if style == "first_dist" else "\n"
    return (f"You are a survey respondent from {resp['country']}. "
            f"These are answers you gave to other questions in the survey:\n"
            f"{profile_lines(resp, q, first_person=True)}\n"
            f"\nFor context, here is how respondents similar to you answered the question "
            f"you are about to answer:{shots_block(nb_idx, q)}"
            f"{extra}"
            f"\nNow tell how you answer this question: {question_for[q]}\nOptions:\n{opts}\n"
            f"Answer as yourself, consistently with your earlier answers. "
            f"Reply with only the number of your chosen option.")

import time
from openai import RateLimitError

def llm_predict(prompt, q, model, max_retries=8):
    labels = labels_for[q]
    extra = ({"chat_template_kwargs": {"enable_thinking": False}}
             if "Qwen3" in model and "2507" not in model else None)
    for attempt in range(max_retries):
        try:
            r = client.chat.completions.create(
                model=model, messages=[{"role": "user", "content": prompt}],
                max_tokens=4, temperature=0, logprobs=True, top_logprobs=20,
                **({"extra_body": extra} if extra else {}))
            break
        except RateLimitError:
            time.sleep(20 * (attempt + 1))   # back off: 20s, 40s, 60s...
    else:
        return None, np.full(len(labels), 1.0 / len(labels))

    probs = np.zeros(len(labels))                     # <-- this block was missing
    try:
        top = r.choices[0].logprobs.content[0].top_logprobs
        for t in top:
            tok = t.token.strip()
            if tok.isdigit() and 1 <= int(tok) <= len(labels):
                probs[int(tok) - 1] += np.exp(t.logprob)
    except Exception:
        pass

    if probs.sum() == 0:   # fallback: parse the generated text
        txt = (r.choices[0].message.content or "").strip()
        for i, l in enumerate(labels):
            if txt.startswith(str(i + 1)) or l.lower() in txt.lower():
                probs[i] = 1.0; break
    if probs.sum() == 0:
        probs[:] = 1.0 / len(labels)
    probs = probs / probs.sum()
    return labels[int(probs.argmax())], probs

## 7. Phase 1 — model × style bake-off (on train, cached)
Six configurations (Qwen3-32B, Qwen3-235B, Llama-3.3-70B × third/first) each predict the same
250 training respondents on all 13 targets; predictions are scored against the known answers
with the official skill formula. Every call is appended to `eval_cache.csv` keyed by
(model, style, respondent, target) and saved every 500 calls, so interruptions and re-runs never
re-pay for completed work (this run shows 0 new calls — everything was already cached).

Reading the comparison table: all six means sit within noise (±0.03) of each other; Qwen3-235B
/ third is the best point estimate (+0.200). Bigger-same-family bought almost nothing over
Qwen3-32B (+0.185), and first-person phrasing did not help on average. Per-target, the LLM is
genuinely strong on Q73 (+0.62) and clearly *below majority guessing* on Q17 and Q112 — the
fact behind the manual overrides in section 8.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

rng = np.random.default_rng(SEED)
per_c = max(1, N_EVAL // train.country.nunique())
eval_pool = (train.groupby("country", group_keys=False)
                  .apply(lambda g: g.sample(min(per_c, len(g)), random_state=SEED))).index
eval_pool = pd.Index(rng.permutation(eval_pool))[:N_EVAL]
compare_idx = eval_pool[:N_COMPARE]

cache = (pd.read_csv(EVAL_CACHE) if os.path.exists(EVAL_CACHE)
         else pd.DataFrame(columns=["model", "style", "idx", "q", "pred", "conf", "probs"]))
def cached_keys():
    return set(zip(cache["model"], cache["style"], cache["idx"], cache["q"]))

def run_eval(model, style, idx_list):
    global cache
    done = cached_keys()
    jobs = [(i, q) for i in idx_list for q in qids
            if (model, style, i, q) not in done and pd.notna(train_lab[q].loc[i])]
    if not jobs:
        return
    print(f"{model} / {style}: {len(jobs)} calls")

    def job(i, q):
        nb = neighbours(train.loc[[i]], q, exclude_self_idx=train.index.get_loc(i))
        pred, probs = llm_predict(build_prompt(train.loc[i], q, nb, style), q, model)
        return {"model": model, "style": style, "idx": i, "q": q, "pred": pred,
                "conf": probs.max(), "probs": json.dumps(probs.tolist())}

    with ThreadPoolExecutor(WORKERS) as pool:
        rows = []
        for k, res in enumerate(pool.map(lambda t: job(*t), jobs)):
            rows.append(res)
            if len(rows) % 500 == 0:
                cache = pd.concat([cache, pd.DataFrame(rows)], ignore_index=True)
                cache.to_csv(EVAL_CACHE, index=False); rows = []
                print(f"  {k+1}/{len(jobs)} done, cache saved")
    if rows:
        cache = pd.concat([cache, pd.DataFrame(rows)], ignore_index=True)
        cache.to_csv(EVAL_CACHE, index=False)

n_calls = len(MODELS) * len(STYLES) * len(compare_idx) * len(qids)
print(f"phase 1 upper bound: {n_calls} calls (cache may reduce this)")
if RUN_LLM_EVAL:
    for m in MODELS:
        for st in STYLES:
            run_eval(m, st, compare_idx)

phase 1 upper bound: 19500 calls (cache may reduce this)


In [ ]:
def config_skill(model, style, idx_list):
    sub = cache[(cache["model"] == model) & (cache["style"] == style) & cache["idx"].isin(idx_list)]
    out = {}
    for q in qids:
        s = sub[sub.q == q].set_index("idx")
        truth = train_lab[q].reindex(s.index).dropna()
        if len(truth) < 30: continue
        s = s.reindex(truth.index)
        maj = (truth == truth.mode()[0]).mean()
        out[q] = skill((s.pred == truth).mean(), maj)
    return out

if RUN_LLM_EVAL:
    tbl = {}
    for m in MODELS:
        for st in STYLES:
            sk = config_skill(m, st, compare_idx)
            if sk: tbl[(m.split('/')[-1], st)] = pd.Series(sk)
    comparison = pd.DataFrame(tbl).round(3)
    comparison.loc["MEAN"] = comparison.mean()
    print(comparison.to_string())

     Qwen3-32B           Qwen3-235B-A22B-Instruct-2507        Llama-3.3-70B-Instruct          
         third     first                         third  first                  third     first
Q201  0.013000  0.000000                     -0.054000  0.040              -0.060000  0.094000
Q73   0.544000  0.620000                      0.620000  0.570               0.525000  0.608000
Q227  0.322000  0.396000                      0.322000  0.356               0.275000  0.275000
Q209  0.288000  0.379000                      0.364000  0.379               0.311000  0.371000
Q33   0.117000  0.160000                      0.142000  0.043               0.130000  0.136000
Q148  0.531000  0.469000                      0.510000  0.517               0.497000  0.524000
Q17  -0.179000 -0.410000                     -0.192000 -0.372              -0.256000 -0.551000
Q186  0.097000  0.032000                      0.177000  0.032               0.113000  0.032000
Q242  0.010000 -0.081000                     -0.02

### Ensemble analysis — free, from the cache
Scores the probability-average of every model pair (same style) using cached vectors — zero new
API calls. Result: no pair beats the best single model (+0.193 vs +0.200), i.e. the three models
make correlated errors on the same hard respondents, so `FINAL_ENSEMBLE` stays off.

In [ ]:
from itertools import combinations

if RUN_LLM_EVAL:
    for st in STYLES:
        for m1, m2 in combinations(MODELS, 2):
            c1 = cache[(cache["model"] == m1) & (cache["style"] == st) & cache["idx"].isin(compare_idx)]
            c2 = cache[(cache["model"] == m2) & (cache["style"] == st) & cache["idx"].isin(compare_idx)]
            merged = c1.merge(c2, on=["idx", "q"], suffixes=("_1", "_2"))
            if merged.empty: continue
            sks = []
            for q in qids:
                s = merged[merged.q == q].set_index("idx")
                truth = train_lab[q].reindex(s.index).dropna()
                if len(truth) < 30: continue
                s = s.reindex(truth.index)
                P = (np.array([json.loads(x) for x in s.probs_1])
                     + np.array([json.loads(x) for x in s.probs_2])) / 2
                pred = [labels_for[q][i] for i in P.argmax(1)]
                maj = (truth == truth.mode()[0]).mean()
                sks.append(skill((pd.Series(pred, index=truth.index) == truth).mean(), maj))
            print(f"{st:10s} {m1.split('/')[-1]} + {m2.split('/')[-1]}: mean skill {np.mean(sks):+.3f}")

third      Qwen3-32B + Qwen3-235B-A22B-Instruct-2507: mean skill +0.193
third      Qwen3-32B + Llama-3.3-70B-Instruct: mean skill +0.183
third      Qwen3-235B-A22B-Instruct-2507 + Llama-3.3-70B-Instruct: mean skill +0.185
first      Qwen3-32B + Qwen3-235B-A22B-Instruct-2507: mean skill +0.184
first      Qwen3-32B + Llama-3.3-70B-Instruct: mean skill +0.187
first      Qwen3-235B-A22B-Instruct-2507 + Llama-3.3-70B-Instruct: mean skill +0.174


## 8. Phase 2 — validate the winner, tune the gate
The winning configuration runs on the full 1,000-respondent evaluation sample (8,301 calls;
the 250 phase-1 respondents are reused from cache). Then, per target, three policies are scored
on these respondents against the *out-of-fold* GBM predictions (so neither expert has seen the
respondent's country): always-GBM, always-LLM, and confidence-gated (GBM when its confidence
≥ t, LLM otherwise, t ∈ {0.4…0.8}). The highest-skill policy is recorded; ties default to GBM.

Result: pure GBM on 5 targets, confidence-gated on 8, pure LLM on none — the LLM earns exactly
the low-confidence slice. The two cells after the printout apply **manual overrides**: Q17 and
Q112 are forced to GBM, because phase 1 showed the LLM with clearly negative standalone skill
there, Q112's gate was tuned on only n=183 respondents (a symptom of the label bug found later),
and both questions have lopsided majorities where wrong deviations are punished hardest.

In [ ]:
if RUN_LLM_EVAL:
    if FINAL_MODEL is None or FINAL_STYLE is None:
        means = {(m, st): np.mean(list(config_skill(m, st, compare_idx).values()) or [-9])
                 for m in MODELS for st in STYLES}
        (FINAL_MODEL, FINAL_STYLE) = max(means, key=means.get)
    print("final config:", FINAL_MODEL, "/", FINAL_STYLE)
    run_eval(FINAL_MODEL, FINAL_STYLE, eval_pool)     # fills the remaining N_EVAL - N_COMPARE
    if FINAL_ENSEMBLE:
        for m in FINAL_ENSEMBLE: run_eval(m, FINAL_STYLE, eval_pool)

final config: Qwen/Qwen3-235B-A22B-Instruct-2507 / third
Qwen/Qwen3-235B-A22B-Instruct-2507 / third: 8301 calls
  500/8301 done, cache saved
  1000/8301 done, cache saved
  1500/8301 done, cache saved
  2000/8301 done, cache saved
  2500/8301 done, cache saved
  3000/8301 done, cache saved
  3500/8301 done, cache saved
  4000/8301 done, cache saved
  4500/8301 done, cache saved
  5000/8301 done, cache saved
  5500/8301 done, cache saved
  6000/8301 done, cache saved
  6500/8301 done, cache saved
  7000/8301 done, cache saved
  7500/8301 done, cache saved
  8000/8301 done, cache saved


In [ ]:
policies = {q: ("gbm", None) for q in qids}
if RUN_LLM_EVAL:
    sub = cache[(cache["model"] == FINAL_MODEL) & (cache["style"] == FINAL_STYLE)
                & cache["idx"].isin(eval_pool)]
    for q in qids:
        s = sub[sub.q == q].set_index("idx")
        truth = train_lab[q].reindex(s.index).dropna()
        s = s.reindex(truth.index)
        g_pred = gbm_oof_pred[q].reindex(truth.index)
        g_prob = gbm_oof_prob[q].reindex(truth.index)
        maj = (truth == truth.mode()[0]).mean()
        def sc(pred): return skill((pred == truth).mean(), maj)
        best = ("gbm", None, sc(g_pred))
        cand = ("llm", None, sc(s.pred))
        if cand[2] > best[2]: best = cand
        for t in [0.4, 0.5, 0.6, 0.7, 0.8]:
            mix = g_pred.where(g_prob >= t, s.pred)
            if sc(mix) > best[2] + 1e-9: best = ("gate", t, sc(mix))
        policies[q] = (best[0], best[1])
        print(f"{q}: {best[0]}{'' if best[1] is None else f' t={best[1]}'} (skill {best[2]:+.3f}, n={len(truth)})")

Q201: gate t=0.4 (skill +0.108, n=991)
Q73: gate t=0.5 (skill +0.616, n=947)
Q227: gate t=0.5 (skill +0.455, n=880)
Q209: gbm (skill +0.461, n=964)
Q33: gate t=0.4 (skill +0.271, n=991)
Q148: gbm (skill +0.556, n=924)
Q17: gate t=0.6 (skill +0.141, n=986)
Q186: gbm (skill +0.316, n=583)
Q242: gate t=0.4 (skill +0.186, n=709)
Q172: gbm (skill +0.278, n=984)
Q22: gate t=0.7 (skill +0.539, n=947)
Q112: gate t=0.6 (skill +0.182, n=183)
Q34: gbm (skill +0.203, n=991)


### Manual policy overrides
Q17 and Q112 are forced to pure GBM. Rationale: phase 1 measured the LLM *below majority
guessing* on both (Q17: −0.18…−0.55 across all six configs); Q112's gate decision rested on
only 183 scoreable respondents; and both targets have dominant majorities, where the skill
metric punishes wrong deviations far more than it rewards correct ones.

In [ ]:
policies["Q17"] = ("gbm", None); policies["Q112"] = ("gbm", None)   # LLM had negative phase-1 skill here
print(policies)

{'Q201': ('gate', 0.4), 'Q73': ('gate', 0.5), 'Q227': ('gate', 0.5), 'Q209': ('gbm', None), 'Q33': ('gate', 0.4), 'Q148': ('gbm', None), 'Q17': ('gbm', None), 'Q186': ('gbm', None), 'Q242': ('gate', 0.4), 'Q172': ('gbm', None), 'Q22': ('gate', 0.7), 'Q112': ('gbm', None), 'Q34': ('gbm', None)}


## 9. Test-time predictions
The full-fit GBM first answers all 13,650 (respondent, target) pairs. The policy table then
routes to the LLM exactly the pairs where the chosen policy demands it — here 1,488 calls
(~11%), i.e. the respondents the GBM is least sure about on gated targets. LLM answers overwrite
the GBM defaults; everything else stands. Output: a complete 13,650-row prediction frame with
no gaps (unanswered pairs would count as wrong).

In [ ]:
final = pd.DataFrame({"respondent_id": np.repeat(test.respondent_id.values, len(qids)),
                      "question_id": qids * len(test)})
pred_map, need_llm = {}, []
for q in qids:
    kind, t = policies[q]
    for i in test.index:
        pred_map[(test.respondent_id[i], q)] = gbm_test.loc[i, q + "_pred"]
        if kind == "llm" or (kind == "gate" and gbm_test.loc[i, q + "_prob"] < t):
            need_llm.append((i, q))
print(f"LLM calls needed on test: {len(need_llm)}"
      + (f" x{len(FINAL_ENSEMBLE)} (ensemble)" if FINAL_ENSEMBLE else ""))

if need_llm and RUN_LLM_TEST:
    models_to_run = list(FINAL_ENSEMBLE) if FINAL_ENSEMBLE else [FINAL_MODEL]
    def tjob(i, q):
        nb = neighbours(test.loc[[i]], q)
        prompt = build_prompt(test.loc[i], q, nb, FINAL_STYLE)
        P = np.mean([llm_predict(prompt, q, m)[1] for m in models_to_run], axis=0)
        return (test.respondent_id[i], q, labels_for[q][int(P.argmax())])
    with ThreadPoolExecutor(WORKERS) as pool:
        for k, (rid, q, pred) in enumerate(pool.map(lambda t: tjob(*t), need_llm)):
            pred_map[(rid, q)] = pred
            if (k + 1) % 250 == 0:
                print(f"  {k+1}/{len(need_llm)} done")
elif need_llm:
    print("RUN_LLM_TEST=False -> keeping GBM answers for these rows.")

final["prediction"] = [pred_map[(r, q)] for r, q in zip(final.respondent_id, final.question_id)]
assert final.prediction.notna().all()
print(len(final), "rows")

LLM calls needed on test: 1488
  250/1488 done
  500/1488 done
  750/1488 done
  1000/1488 done
  1250/1488 done
13650 rows


## 10. Calibration check, the Q112 incident, and the submission
**Calibration check.** For the 20 seen countries we compare the predicted answer distribution
per (target, country) against the training distribution, as a total-variation distance. With 30
test respondents per country, 0.15–0.25 is ordinary sampling noise; large values flag either
argmax mode-collapse (expected, costs Alignment but wins Skill) or real bugs.

**The Q112 incident.** The first pass showed Q112 at 0.36 with Peru at 0.97 — near-total
disagreement. The diagnostic cell revealed the cause: train stores Q112 as a raw 1–10 corruption
scale, while the targets define 5 binned labels; naive code=option mapping had silently kept
only ~19% of training answers, mislabeled. The fix cell re-bins 1–10 into the five labels
(1-2→No corruption … 9-10→Very high), restoring 98% coverage, retrains Q112's GBM, refreshes its
test predictions, and pins its policy to GBM. Post-fix Q112 calibration: 0.21 — normal. The
remaining worst cells (Q186-Germany etc.) are the known argmax-vs-distribution trade-off, not
errors.

**Submission.** Writes the three required parts — `predictions.csv` (13,650 rows),
`features.csv` (the full pool declared per target, since the GBM consumes every variable), and
`method/` (an example prompt per target with its policy, plus a method note that documents the
overrides) — validates row count, respondent coverage, non-null predictions and exact label
strings, then zips `submission/` for upload.

In [ ]:
rows = []
for q in qids:
    for c in seen:
        p_train = train_lab.loc[train.country == c, q].value_counts(normalize=True)
        rids = test.loc[test.country == c, "respondent_id"]
        p_pred = (final[(final.question_id == q) & final.respondent_id.isin(rids)]
                  .prediction.value_counts(normalize=True))
        idx = p_train.index.union(p_pred.index)
        rows.append({"q": q, "country": c,
                     "tv": 0.5 * (p_train.reindex(idx, fill_value=0)
                                  - p_pred.reindex(idx, fill_value=0)).abs().sum()})
cal = pd.DataFrame(rows)
print(cal.groupby("q").tv.mean().round(3))
print("\nworst 5:"); print(cal.nlargest(5, "tv").to_string(index=False))

q
Q112    0.213
Q148    0.188
Q17     0.096
Q172    0.231
Q186    0.288
Q201    0.192
Q209    0.107
Q22     0.057
Q227    0.148
Q242    0.284
Q33     0.191
Q34     0.255
Q73     0.125
Name: tv, dtype: float64

worst 5:
   q    country       tv
Q186    Germany 0.661905
Q242 Bangladesh 0.566667
Q227      China 0.500000
Q148      Egypt 0.500000
Q148 Kazakhstan 0.500000


### Diagnosing the Q112 calibration outlier
Q112's mean TV of 0.36 (Peru: 0.97) demanded a look at the raw data: train's answer rate is 98%,
yet only 3 Peruvian answers survived our mapping — proof that the code→label mapping, not the
model, was broken.

In [ ]:
q, c = "Q112", "Peru"
print("train answer rate:", train[q].notna().mean().round(2))
print("\ntrain Peru answers:")
print(train_lab.loc[train.country == c, q].value_counts())
print("\nour Peru predictions:")
rids = test.loc[test.country == c, "respondent_id"]
print(final[(final.question_id == q) & final.respondent_id.isin(rids)].prediction.value_counts())

train answer rate: 0.98

train Peru answers:
Q112
No corruption                   2
Moderate level of corruption    1
Name: count, dtype: int64

our Peru predictions:
prediction
Very high corruption    29
No corruption            1
Name: count, dtype: int64


### Repairing Q112
Train stores Q112 as a raw 1–10 scale; the targets define 5 binned labels. The fix maps 1–2 →
"No corruption" … 9–10 → "Very high corruption", rebuilds the truth column (98% mapped),
retrains the target's GBM, refreshes its test predictions, and keeps the policy at GBM (the LLM
evaluation for this target had used the broken truth). The prediction spread printed at the end
replaces the earlier degenerate output.

In [ ]:
# --- FIX Q112: train stores raw 1-10 scale; targets define binned labels ---
q = "Q112"
print(targets[targets.question_id == q][["option", "label"]])   # confirm option count!

# raw 1-10 -> 5 bins: 1-2 ->1, 3-4 ->2, 5-6 ->3, 7-8 ->4, 9-10 ->5  (adjust if not 5 options)
bin_map = {c: (c + 1) // 2 for c in range(1, 11)}
train_lab[q] = train[q].map(lambda v: opt2lab[q].get(bin_map[int(v)]) if pd.notna(v) else np.nan)
print("mapped share now:", train_lab[q].notna().mean().round(2))   # should be ~0.98

# retrain Q112's GBM on the correctly-labeled data and refresh test predictions
y = train_lab[q]; mask = y.notna()
X = train.loc[mask, COLS].astype(float); yy = y[mask].values
gbm_full[q] = new_model().fit(X, yy)
P = gbm_full[q].predict_proba(Xte)
gbm_test[q + "_pred"] = gbm_full[q].classes_[P.argmax(1)]
gbm_test[q + "_prob"] = P.max(1)
policies[q] = ("gbm", None)          # keep GBM-only here (LLM eval used the broken truth)

# overwrite Q112 in the final predictions
for i in test.index:
    pred_map[(test.respondent_id[i], q)] = gbm_test.loc[i, q + "_pred"]
final["prediction"] = [pred_map[(r, qq)] for r, qq in zip(final.respondent_id, final.question_id)]
print(final[final.question_id == q].prediction.value_counts())

    option                         label
48       1                 No corruption
49       2                Low corruption
50       3  Moderate level of corruption
51       4               High corruption
52       5          Very high corruption
mapped share now: 0.98
prediction
Very high corruption            592
High corruption                 277
Moderate level of corruption    131
Low corruption                   33
No corruption                    17
Name: count, dtype: int64


In [ ]:
from pathlib import Path
import shutil

sub = Path("submission"); (sub / "method").mkdir(parents=True, exist_ok=True)
final.to_csv(sub / "predictions.csv", index=False)
pd.DataFrame([{"question_id": q, "feature_variable_code": v}
              for q in qids for v in fv]).to_csv(sub / "features.csv", index=False)

_model_desc = (" + ".join(FINAL_ENSEMBLE) if FINAL_ENSEMBLE else str(FINAL_MODEL))
example_prompts = {q: build_prompt(test.iloc[0], q, neighbours(test.iloc[[0]], q),
                                   FINAL_STYLE or "third") for q in qids}
(sub / "method" / "prompts.jsonl").write_text(
    "\n".join(json.dumps({"question_id": q, "model": _model_desc,
                           "style": str(FINAL_STYLE), "policy": str(policies[q]),
                           "example_prompt": p}) for q, p in example_prompts.items()),
    encoding="utf-8")
(sub / "method" / "method.md").write_text(
    "Gated ensemble. Primary: per-target HistGradientBoosting on all pool features "
    "(ordinal codes, native NaN handling), leave-countries-out validated. Challenger: "
    f"{_model_desc}, prompt style '{FINAL_STYLE}', per-target top-{TOP_FEATURES} features "
    f"(bias-corrected Cramér's V), {K_SHOTS} V-weighted kNN few-shot train examples, numbered "
    "options scored via first-token logprobs. Model and prompt style selected on a held-out "
    "train comparison sample; per-target policy (GBM/LLM/confidence-gated) tuned on a larger "
    "disjointly-scored train sample using the skill metric.\n"
    "Manual override: Q17 and Q112 forced to the GBM policy, as the LLM showed"
    "negative standalone skill on these targets in offline evaluation.\n", encoding="utf-8")

p = pd.read_csv(sub / "predictions.csv")
assert len(p) == len(test) * len(qids), f"row count {len(p)}"
assert set(p.respondent_id) == set(test.respondent_id)
assert p.prediction.notna().all()
valid = set(map(tuple, targets[["question_id", "label"]].values))
bad = [t for t in map(tuple, p[["question_id", "prediction"]].values) if t not in valid]
assert not bad, bad[:5]
print("submission format OK:", len(p), "rows")
shutil.make_archive("submission", "zip", ".", "submission")
print("wrote submission.zip")

submission format OK: 13650 rows
wrote submission.zip
